# Test results

The test split, evaluated once with the models, thresholds and analyses fixed in `reports/test_plan.md`
(committed before the split was opened). Neural-model predictions come from the Colab cell *Test predictions*;
download `MyDrive/doc_verify/test_predictions/*.csv` into `models/test_predictions/` first.

In [ ]:
import json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, f1_score

from doc_verify.config import FIGURES_DIR, MODELS_DIR, ROOT
from doc_verify.data import KIND_CLASSES, MISSING, TYPE_CLASSES, all_pages
from doc_verify.evaluate import (
    auth_breakdown,
    mcnemar,
    pair_bootstrap_auc,
    pair_table,
    type_bootstrap_f1,
    type_confusion,
)

pages = all_pages()
thresholds = json.loads((ROOT / 'reports' / 'val_thresholds.json').read_text())
NAMES = {
    'fused': 'Fused (main)', 'fused_no_kind': 'Fused, λ₂ = 0', 'image_vit': 'Image-only ViT',
    'resnet18': 'ResNet18 scratch', 'tfidf': 'TF-IDF + LR', 'clip_zero_shot': 'CLIP zero-shot',
    'rule_checker': 'Rule checker',
}
predictions = {name: pd.read_csv(MODELS_DIR / 'test_predictions' / f'{name}.csv') for name in NAMES}
test = pages[pages.split == 'test']
for name, p in predictions.items():
    assert p.page_index.isin(test.page_index).all(), name
    for column in ('type_label', 'auth_label', 'kind_label'):
        if column in p:
            assert (p[column].to_numpy() == test.set_index('page_index').loc[p.page_index, column].to_numpy()).all()
print({name: len(p) for name, p in predictions.items()})

## 1. Document type (Tobacco3482 test)

In [ ]:
TYPE_MODELS = ['fused', 'fused_no_kind', 'image_vit', 'resnet18', 'tfidf', 'clip_zero_shot']
rows = []
for name in TYPE_MODELS:
    typed = predictions[name][predictions[name].type_label != MISSING]
    f1, low, high = type_bootstrap_f1(predictions[name])
    rows.append({'model': NAMES[name], 'pages': len(typed),
                 'accuracy': accuracy_score(typed.type_label, typed.type_pred),
                 'macro_f1': f1, 'f1_ci_low': low, 'f1_ci_high': high})
type_table = pd.DataFrame(rows).set_index('model')
type_table.round(3)

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 10), constrained_layout=True)
short = ['advert', 'email', 'form', 'letter', 'memo', 'news', 'note', 'report', 'resume', 'scientific']
for ax, name in zip(axes.flat, TYPE_MODELS):
    counts = type_confusion(predictions[name])
    share = counts.div(counts.sum(axis=1), axis=0)
    ax.imshow(share, cmap='Blues', vmin=0, vmax=1)
    for i in range(len(TYPE_CLASSES)):
        for j in range(len(TYPE_CLASSES)):
            if counts.iat[i, j]:
                ax.text(j, i, counts.iat[i, j], ha='center', va='center', fontsize=8,
                        color='white' if share.iat[i, j] > 0.55 else '#333333')
    ax.set_title(f"{NAMES[name]} (macro-F1 {type_table.loc[NAMES[name], 'macro_f1']:.3f})", fontsize=11)
    ax.set_xticks(range(len(short)), short, rotation=45, fontsize=8)
    ax.set_yticks(range(len(short)), short, fontsize=8)
    ax.set_xlabel('predicted', fontsize=9)
    ax.set_ylabel('true', fontsize=9)
fig.suptitle('Document type on the Tobacco3482 test split: counts, shaded by share of the true class', fontsize=12)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
fig.savefig(FIGURES_DIR / 'test_type_confusion.png', dpi=200)
plt.show()

## 2. Authenticity and kind (SAD-Syn test)
AUC intervals resample clean/altered pairs; accuracy and macro-F1 use the validation thresholds.

In [ ]:
AUTH_MODELS = ['fused', 'fused_no_kind', 'image_vit', 'tfidf', 'rule_checker']
rows = []
for name in AUTH_MODELS:
    authed = predictions[name][predictions[name].auth_label != MISSING]
    called = (authed.auth_score >= thresholds[name]).astype(int)
    auc, low, high = pair_bootstrap_auc(pair_table(predictions[name], pages))
    rows.append({'model': NAMES[name], 'pages': len(authed), 'auc': auc, 'auc_ci_low': low, 'auc_ci_high': high,
                 'threshold': thresholds[name],
                 'accuracy': accuracy_score(authed.auth_label, called),
                 'macro_f1': f1_score(authed.auth_label, called, average='macro'),
                 'kind_macro_f1': f1_score(authed.kind_label, authed.kind_pred, average='macro',
                                           labels=range(len(KIND_CLASSES)), zero_division=0)})
auth_table = pd.DataFrame(rows).set_index('model')
auth_table.round(3)

## 3. Authenticity by kind, document type and evidence readability

In [ ]:
breakdown = pd.concat([auth_breakdown(predictions[name], pages).assign(model=NAMES[name]) for name in AUTH_MODELS])
breakdown['auc (95% CI)'] = [f'{a:.3f} ({lo:.2f}-{hi:.2f})' for a, lo, hi in zip(breakdown.auc, breakdown.ci_low, breakdown.ci_high)]
pairs = breakdown.drop_duplicates(['group', 'value']).set_index(['group', 'value']).pairs
breakdown.pivot_table(index=['group', 'value'], columns='model', values='auc (95% CI)', aggfunc='first',
                      sort=False)[[NAMES[n] for n in AUTH_MODELS]].join(pairs)

## 4. McNemar tests (exact, Holm-corrected across the three)

In [ ]:
def correct_type(name):
    p = predictions[name][predictions[name].type_label != MISSING].set_index('page_index').sort_index()
    return p.type_pred == p.type_label


def correct_auth(name):
    p = predictions[name][predictions[name].auth_label != MISSING].set_index('page_index').sort_index()
    return (p.auth_score >= thresholds[name]).astype(int) == p.auth_label


tests = [
    ('type', 'fused', 'image_vit', correct_type),
    ('authenticity', 'fused', 'rule_checker', correct_auth),
    ('type', 'fused', 'fused_no_kind', correct_type),
]
rows = []
for task, a, b, correct in tests:
    right_a, right_b = correct(a), correct(b)
    assert right_a.index.equals(right_b.index)
    rows.append({'task': task, 'model_a': NAMES[a], 'model_b': NAMES[b], 'pages': len(right_a),
                 'acc_a': right_a.mean(), 'acc_b': right_b.mean(), **mcnemar(right_a, right_b)})
mcnemar_table = pd.DataFrame(rows)

# Holm: the k-th smallest p-value is multiplied by (m - k + 1), kept monotone, capped at 1
order = np.argsort(mcnemar_table.p_value.to_numpy())
holm = np.empty(len(order))
running = 0.0
for rank, i in enumerate(order):
    running = max(running, (len(order) - rank) * mcnemar_table.p_value.iloc[i])
    holm[i] = min(1.0, running)
mcnemar_table['p_holm'] = holm
mcnemar_table.round(4)

In [ ]:
out = ROOT / 'reports'
type_table.to_csv(out / 'test_type.csv')
auth_table.to_csv(out / 'test_authenticity.csv')
breakdown.drop(columns='auc (95% CI)').to_csv(out / 'test_auth_breakdown.csv', index=False)
mcnemar_table.to_csv(out / 'test_mcnemar.csv', index=False)
print('saved to', out)